In [ ]:
import pandas as pd
import numpy as np


In [ ]:
!head -n 5 /content/ner_disaster_india_1000.jsonl.csv


"id"	"text"	"entities"
"1"	"A orange alert dust storm was reported in Aizawl on 30 September 2021 23:00."	"start:15|end:25|label:DISASTER_TYPE
start:42|end:48|label:LOCATION
start:52|end:75|label:DATE_TIME"
"2"	"IMD reported a cyclone of 168 kmph winds in Dadra and Nagar Haveli on 15 September 2023 18:30."	"start:0|end:3|label:ORGANIZATION


In [ ]:
import pandas as pd

df = pd.read_csv('/content/ner_disaster_india_1000.jsonl.csv', sep='\t')
print(df.head())

   id                                               text  \
0   1  A orange alert dust storm was reported in Aiza...   
1   2  IMD reported a cyclone of 168 kmph winds in Da...   
2   3  Municipal Corporation reported a drought of in...   
3   4  Following the hailstorm in Chandigarh on 22 Ju...   
4   5  A moderate flood was reported in Kota on 03 Fe...   

                                            entities  
0  start:15|end:25|label:DISASTER_TYPE\nstart:42|...  
1  start:0|end:3|label:ORGANIZATION\nstart:15|end...  
2  start:0|end:21|label:ORGANIZATION\nstart:33|en...  
3  start:14|end:23|label:DISASTER_TYPE\nstart:27|...  
4  start:11|end:16|label:DISASTER_TYPE\nstart:33|...  


In [ ]:
df.shape

(1000, 3)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        1000 non-null   int64 
 1   text      1000 non-null   object
 2   entities  1000 non-null   object
dtypes: int64(1), object(2)
memory usage: 23.6+ KB


In [ ]:
def parse_entities(ent_str):
    if pd.isna(ent_str):
        return []
    entities = []
    for line in ent_str.strip().split('\n'):
        parts = line.split('|')
        entity = {}
        for p in parts:
            key, value = p.split(':', 1)
            if key in ['start', 'end']:
                entity[key] = int(value)
            else:
                entity[key] = value
        entities.append(entity)
    return entities

df['labels'] = df['entities'].apply(parse_entities)
df = df[['text', 'labels']]  # Keep only what we need
print(df.iloc[0])

text      A orange alert dust storm was reported in Aiza...
labels    [{'start': 15, 'end': 25, 'label': 'DISASTER_T...
Name: 0, dtype: object


In [ ]:
df.head()

,text,labels
0,A orange alert dust storm was reported in Aiza...,"[{'start': 15, 'end': 25, 'label': 'DISASTER_T..."
1,IMD reported a cyclone of 168 kmph winds in Da...,"[{'start': 0, 'end': 3, 'label': 'ORGANIZATION..."
2,Municipal Corporation reported a drought of in...,"[{'start': 0, 'end': 21, 'label': 'ORGANIZATIO..."
3,Following the hailstorm in Chandigarh on 22 Ju...,"[{'start': 14, 'end': 23, 'label': 'DISASTER_T..."
4,A moderate flood was reported in Kota on 03 Fe...,"[{'start': 11, 'end': 16, 'label': 'DISASTER_T..."


In [ ]:
!pip install -q transformers datasets seqeval evaluate
import ast
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
import evaluate
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
def tokenize_and_align_labels(example):
    text = example["text"]
    entities = example["labels"]
    tokenized = tokenizer(text, return_offsets_mapping=True, truncation=True)
    labels = ["O"] * len(tokenized["offset_mapping"])

    for ent in entities:
        for i, (start, end) in enumerate(tokenized["offset_mapping"]):
            if start >= ent["start"] and end <= ent["end"]:
                labels[i] = "B-" + ent["label"] if start == ent["start"] else "I-" + ent["label"]
    tokenized["labels"] = labels
    return tokenized

In [ ]:
tokenized_data = [tokenize_and_align_labels(row) for _, row in df.iterrows()]

In [ ]:
unique_labels = sorted(list(set(lbl for x in tokenized_data for lbl in x["labels"])))
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

In [ ]:
def convert_to_hf_format(data):
    tokens = [d["input_ids"] for d in data]
    attention_masks = [d["attention_mask"] for d in data]
    label_ids = []
    for d in data:
        label_ids.append([label2id[lbl] for lbl in d["labels"]])
    return Dataset.from_dict({
        "input_ids": tokens,
        "attention_mask": attention_masks,
        "labels": label_ids
    })

dataset = convert_to_hf_format(tokenized_data)
dataset_split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id,
)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments, __version__ as tfm_version
import inspect, transformers

print("transformers version:", tfm_version)
print("TrainingArguments module:", TrainingArguments.__module__)
print("TrainingArguments source file:", inspect.getsourcefile(TrainingArguments))
print("TrainingArguments signature:", inspect.signature(TrainingArguments.__init__))

transformers version: 4.57.1
TrainingArguments module: transformers.training_args
TrainingArguments source file: /usr/local/lib/python3.12/dist-packages/transformers/training_args.py
TrainingArguments signature: (self, output_dir: Optional[str] = None, overwrite_output_dir: bool = False, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: Union[transformers.trainer_utils.IntervalStrategy, str] = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, per_gpu_train_batch_size: Optional[int] = None, per_gpu_eval_batch_size: Optional[int] = None, gradient_accumulation_steps: int = 1, eval_accumulation_steps: Optional[int] = None, eval_delay: float = 0, torch_empty_cache_steps: Optional[int] = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, m

In [ ]:
args = TrainingArguments(
    output_dir="disaster_ner_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
)

In [ ]:
metric = evaluate.load("seqeval")

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[pred] for (pred, lab) in zip(pred_row, label_row) if lab != -100]
        for pred_row, label_row in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[lab] for (pred, lab) in zip(pred_row, label_row) if lab != -100]
        for pred_row, label_row in zip(predictions, labels)
    ]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

/tmp/ipython-input-2178614684.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: parthdubey118 (parthdubey118-shri-govindram-seksariya-institute-of-tech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.017463,0.990982,0.993970,0.992474,0.997368
2,No log,0.002473,1.000000,1.000000,1.000000,1.000000
3,No log,0.001772,1.000000,1.000000,1.000000,1.000000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=300, training_loss=0.18432684580485026, metrics={'train_runtime': 867.2885, 'train_samples_per_second': 2.767, 'train_steps_per_second': 0.346, 'total_flos': 31272982609104.0, 'train_loss': 0.18432684580485026, 'epoch': 3.0})

In [ ]:
trainer.save_model("final_disaster_ner_model")

In [ ]:
test_text = "Over 500 people killed in just last 3 weeks in bus accidents and fire Now back to back two train accidents Yesterday, half a dozen women killed by a train in UP Now it a dozen killed in Chhattisgarh  And there will be no accountability."
inputs = tokenizer(test_text, return_tensors="pt")
outputs = model(**inputs)
predictions = outputs.logits.argmax(-1).squeeze().tolist()
predicted_labels = [id2label[i] for i in predictions]
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze())
for token, label in zip(tokens, predicted_labels):
    print(f"{token}: {label}")

[CLS]: B-DISASTER_TYPE
Over: O
500: I-CASUALTIES
people: I-CASUALTIES
killed: O
in: O
just: O
last: O
3: I-LOCATION
weeks: I-DATE_TIME
in: O
bus: B-DISASTER_TYPE
accidents: I-DISASTER_TYPE
and: O
fire: B-DISASTER_TYPE
Now: O
back: O
to: O
back: O
two: O
train: B-DISASTER_TYPE
accidents: I-DISASTER_TYPE
Yesterday: O
,: O
half: O
a: I-CASUALTIES
dozen: I-CASUALTIES
women: B-DISASTER_TYPE
killed: O
by: O
a: O
train: B-DISASTER_TYPE
in: O
UP: B-LOCATION
Now: O
it: O
a: O
dozen: I-DISASTER_TYPE
killed: I-CASUALTIES
in: O
Ch: B-LOCATION
##hat: I-LOCATION
##tis: I-LOCATION
##garh: I-LOCATION
And: O
there: O
will: O
be: O
no: O
account: O
##ability: O
.: O
[SEP]: B-DISASTER_TYPE


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# load the fine-tuned model you trained earlier
model_path = "/content/final_disaster_ner_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

# create NER pipeline
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

Device set to use cpu


In [ ]:
df_new = pd.read_csv("/texts1.csv")


In [ ]:
df_new.head()

,ये #earthquake भी बिन मौसम बारिश जैसे हो गई है #Delhi में कभी भी आ जा रही है,"Mar 21, 2023",10:25 PM,https://twitter.com/pallavi0305/status/1638222884350099456,1
0,Never felt such a strong earthquake ever. #Dha...,"Jan 10, 2022",1:44 AM,https://twitter.com/sportyrajesh/status/148027...,1
1,"I didnt experience Nepal Earthquake. BTW, 3.5 ...","Jan 10, 2022",2:18 PM,https://twitter.com/sportyrajesh/status/148046...,1
2,"Earthquake tremors felt in Delhi, Gurugram and...","Jul 5, 2021",11:26 PM,https://twitter.com/24TopNews_/status/14121080...,1
3,#earthquake A 6.7-magnitude earthquake hit Nor...,"Apr 30, 2021",11:55 PM,https://twitter.com/24TopNews_/status/13881977...,1
4,Earthquake of magnitude 3.0 occurred at 06:09:...,"Jun 21, 2021",6:43 AM,https://twitter.com/24TopNews_/status/14067822...,1


In [ ]:
# Rename the columns using the first row as headers
new_columns = df_new.iloc[0].tolist()
df_new.columns = new_columns
df_new = df_new[1:].reset_index(drop=True)

# Rename the first column to 'text'
df_new.rename(columns={df_new.columns[0]: 'text'}, inplace=True)

display(df_new.head())

,text,"Jan 10, 2022",2:18 PM,https://twitter.com/sportyrajesh/status/1480461592613437442,1
0,"Earthquake tremors felt in Delhi, Gurugram and...","Jul 5, 2021",11:26 PM,https://twitter.com/24TopNews_/status/14121080...,1
1,#earthquake A 6.7-magnitude earthquake hit Nor...,"Apr 30, 2021",11:55 PM,https://twitter.com/24TopNews_/status/13881977...,1
2,Earthquake of magnitude 3.0 occurred at 06:09:...,"Jun 21, 2021",6:43 AM,https://twitter.com/24TopNews_/status/14067822...,1
3,Earthquake of magnitude 2.6 hits #DelhiNCR #De...,"Nov 11, 2023",4:20 PM,https://twitter.com/dna/status/172329217025238...,1
4,#Nepal में फिर भूकंप के तेज झटके 5.6 तीव्रता ज...,"Nov 6, 2023",4:35 PM,https://twitter.com/AG_Journalist/status/17214...,1


In [ ]:
df_new = df_new[['text']]
display(df_new.head())

,text
0,"Earthquake tremors felt in Delhi, Gurugram and..."
1,#earthquake A 6.7-magnitude earthquake hit Nor...
2,Earthquake of magnitude 3.0 occurred at 06:09:...
3,Earthquake of magnitude 2.6 hits #DelhiNCR #De...
4,#Nepal में फिर भूकंप के तेज झटके 5.6 तीव्रता ज...


In [ ]:
import re

def clean_text(text):
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    # Remove special characters (keeping letters, numbers, and some basic punctuation)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?;:]', '', text)
    return text

df_new['cleaned_text'] = df_new['text'].apply(clean_text)
display(df_new.head())

,text,cleaned_text
0,"Earthquake tremors felt in Delhi, Gurugram and...","Earthquake tremors felt in Delhi, Gurugram and..."
1,#earthquake A 6.7-magnitude earthquake hit Nor...,earthquake A 6.7magnitude earthquake hit North...
2,Earthquake of magnitude 3.0 occurred at 06:09:...,Earthquake of magnitude 3.0 occurred at 06:09:...
3,Earthquake of magnitude 2.6 hits #DelhiNCR #De...,Earthquake of magnitude 2.6 hits DelhiNCR Delh...
4,#Nepal में फिर भूकंप के तेज झटके 5.6 तीव्रता ज...,Nepal 5.6 35 km Delhi ncr De...


In [ ]:
df_new = df_new.drop('text', axis=1)
df_new = df_new.rename(columns={'cleaned_text': 'text'})
display(df_new.head())

,text
0,"Earthquake tremors felt in Delhi, Gurugram and..."
1,earthquake A 6.7magnitude earthquake hit North...
2,Earthquake of magnitude 3.0 occurred at 06:09:...
3,Earthquake of magnitude 2.6 hits DelhiNCR Delh...
4,Nepal 5.6 35 km Delhi ncr De...


In [ ]:
texts = df_new["text"].tolist()

In [ ]:
predictions = []

for text in texts:
    ents = ner_pipeline(text)
    formatted = [
        {
            "entity": e["entity_group"],
            "word": e["word"],
            "start": e["start"],
            "end": e["end"],
            "score": round(e["score"], 3)
        }
        for e in ents
    ]
    predictions.append(formatted)

In [ ]:
df_new["predicted_entities"] = predictions
df_new.to_csv("labeled_texts.csv", index=False)
print("✅ Saved labeled file as labeled_texts.csv")

✅ Saved labeled file as labeled_texts.csv
